# 📈 Stock Price Prediction Using Deep Learning
### A Comparative Analysis of LSTM, GRU, BiLSTM, and BiGRU Models
**VIT Bhopal University | Group 142**

**Team:** Mohit Kumar Mishra, Charchit Bari, Anshika Kumari, Siddhi Sibangi Kar, Anushka Thakur, Vishal Dubey

**Supervisor:** Dr. Anil Kumar Yadav

## 1. Install & Import Libraries

In [ ]:
# Install required libraries
!pip install yfinance scikit-learn tensorflow matplotlib seaborn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Bidirectional, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

print('TensorFlow version:', tf.__version__)
print('All libraries loaded successfully ✅')

## 2. Data Acquisition

In [ ]:
# Fetch Apple (AAPL) historical data from Yahoo Finance
ticker = 'AAPL'
start_date = '2020-01-01'
end_date   = '2024-01-01'

df = yf.download(ticker, start=start_date, end=end_date)
df.reset_index(inplace=True)
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Plot historical closing price
plt.figure(figsize=(14, 5))
plt.plot(df['Date'], df['Close'], color='royalblue', linewidth=1.5)
plt.title(f'{ticker} Closing Price (2020–2024)', fontsize=15)
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/price_chart.png', dpi=150)
plt.show()

## 3. Data Preprocessing

In [ ]:
# Use only Close price
data = df[['Close']].values

# Normalize with MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

# Train/test split (80/20)
train_size = int(len(scaled_data) * 0.8)
train_data = scaled_data[:train_size]
test_data  = scaled_data[train_size:]

print(f'Training samples : {len(train_data)}')
print(f'Testing  samples : {len(test_data)}')

# Create sliding-window sequences (look-back = 60 days)
def create_sequences(data, look_back=60):
    X, y = [], []
    for i in range(look_back, len(data)):
        X.append(data[i - look_back:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

LOOK_BACK = 60
X_train, y_train = create_sequences(train_data, LOOK_BACK)
X_test,  y_test  = create_sequences(
    np.concatenate([train_data[-LOOK_BACK:], test_data]), LOOK_BACK
)

# Reshape for RNN: (samples, time_steps, features)
X_train = X_train.reshape(-1, LOOK_BACK, 1)
X_test  = X_test.reshape(-1, LOOK_BACK, 1)

print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}  | y_test : {y_test.shape}')

## 4. Model Definitions

In [ ]:
def build_lstm(input_shape):
    model = Sequential([
        LSTM(128, return_sequences=True, input_shape=input_shape),
        Dropout(0.3),
        LSTM(64),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.0001), loss='mse', metrics=['mae'])
    return model

def build_gru(input_shape):
    model = Sequential([
        GRU(128, return_sequences=True, input_shape=input_shape),
        Dropout(0.3),
        GRU(64),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.0001), loss='mse', metrics=['mae'])
    return model

def build_bilstm(input_shape):
    model = Sequential([
        Bidirectional(LSTM(128, return_sequences=True), input_shape=input_shape),
        Dropout(0.3),
        Bidirectional(LSTM(64)),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.0001), loss='mse', metrics=['mae'])
    return model

def build_bigru(input_shape):
    model = Sequential([
        Bidirectional(GRU(128, return_sequences=True), input_shape=input_shape),
        Dropout(0.3),
        Bidirectional(GRU(64)),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.0001), loss='mse', metrics=['mae'])
    return model

input_shape = (LOOK_BACK, 1)
print('Model builders ready ✅')

## 5. Training All Models

In [ ]:
EPOCHS     = 50
BATCH_SIZE = 32

callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.5, patience=5, monitor='val_loss')
]

model_builders = {
    'LSTM'  : build_lstm,
    'GRU'   : build_gru,
    'BiLSTM': build_bilstm,
    'BiGRU' : build_bigru,
}

trained_models  = {}
training_histories = {}

for name, builder in model_builders.items():
    print(f'\n🔄 Training {name}...')
    model = builder(input_shape)
    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.1,
        callbacks=callbacks,
        verbose=1
    )
    trained_models[name]      = model
    training_histories[name]  = history
    print(f'✅ {name} training complete!')

## 6. Evaluation & Comparison

In [ ]:
def evaluate_model(model, X_test, y_test, scaler):
    preds_scaled = model.predict(X_test, verbose=0)
    preds = scaler.inverse_transform(preds_scaled)
    actual = scaler.inverse_transform(y_test.reshape(-1, 1))
    rmse = np.sqrt(mean_squared_error(actual, preds))
    mae  = mean_absolute_error(actual, preds)
    r2   = r2_score(actual, preds)
    mape = np.mean(np.abs((actual - preds) / actual)) * 100
    acc  = 100 - mape
    return {'RMSE': round(rmse,2), 'MAE': round(mae,2),
            'R2': round(r2,4), 'MAPE': round(mape,2),
            'Accuracy': round(acc,2), 'predictions': preds, 'actual': actual}

results = {}
for name, model in trained_models.items():
    results[name] = evaluate_model(model, X_test, y_test, scaler)
    print(f"{name:8s} | Acc: {results[name]['Accuracy']}% | RMSE: {results[name]['RMSE']} | MAE: {results[name]['MAE']} | R²: {results[name]['R2']}")

# Summary table
summary = pd.DataFrame({k: {m: v for m,v in r.items() if m not in ['predictions','actual']}
                        for k, r in results.items()}).T
print('\n📊 Model Comparison Table:')
print(summary.to_string())

## 7. Visualizations

In [ ]:
# --- Actual vs Predicted (BiLSTM best model) ---
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(results['BiLSTM']['actual'], label='Actual Price', color='royalblue', linewidth=1.5)
ax.plot(results['BiLSTM']['predictions'], label='Predicted Price', color='crimson',
        linestyle='--', linewidth=1.5)
ax.set_title('BiLSTM — Actual vs Predicted Stock Price', fontsize=14)
ax.set_xlabel('Time Steps')
ax.set_ylabel('Price ($)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/bilstm_prediction.png', dpi=150)
plt.show()

In [ ]:
# --- Error Distribution (all models) ---
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
colors = {'LSTM':'steelblue','GRU':'green','BiLSTM':'crimson','BiGRU':'orange'}
for ax, (name, res) in zip(axes, results.items()):
    errors = res['actual'].flatten() - res['predictions'].flatten()
    ax.hist(errors, bins=40, color=colors[name], alpha=0.8, edgecolor='white')
    ax.axvline(0, color='black', linestyle='--')
    ax.set_title(f'{name}\nAcc: {res["Accuracy"]}% | RMSE: {res["RMSE"]}')
    ax.set_xlabel('Prediction Error')
axes[0].set_ylabel('Frequency')
plt.suptitle('Error Distribution Comparison Across Models', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Training Loss Curves ---
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, history) in zip(axes, training_histories.items()):
    ax.plot(history.history['loss'],     label='Train Loss', color='royalblue')
    ax.plot(history.history['val_loss'], label='Val Loss',   color='crimson', linestyle='--')
    ax.set_title(f'{name} — Loss Curve')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.suptitle('Training & Validation Loss', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Bar Chart Comparison ---
metrics = ['Accuracy', 'RMSE', 'MAE']
model_names = list(results.keys())
bar_colors = ['steelblue', 'green', 'crimson', 'orange']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric in zip(axes, metrics):
    vals = [results[m][metric] for m in model_names]
    bars = ax.bar(model_names, vals, color=bar_colors, edgecolor='white', width=0.5)
    ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=10)
    ax.set_title(f'Model Comparison — {metric}', fontsize=12)
    ax.set_ylabel(metric)
    ax.grid(axis='y', alpha=0.3)
plt.suptitle('Model Performance Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('../results/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best = max(results, key=lambda x: results[x]['Accuracy'])
print(f'\n🏆 Best Model: {best} with {results[best]["Accuracy"]}% accuracy')